# 🧠 EX47: คะแนนความมั่นใจ (Confidence Score) ในการตรวจจับวัตถุ
### *การบรรยายโดยศาสตราจารย์ด้านคอมพิวเตอร์วิทัศน์ของคุณ*

ยินดีต้อนรับนักเรียนทุกคนกลับเข้าสู่บทเรียน! วันนี้เราจะมาเจาะลึกองค์ประกอบที่สำคัญที่สุดอย่างหนึ่งของการตรวจจับวัตถุ นั่นคือ **คะแนนความมั่นใจ (Confidence Scores)**

ในการบรรยายครั้งนี้ เราจะ:
1. วิเคราะห์สูตรคะแนนความมั่นใจแบบ anchor-based ของ **YOLO ดั้งเดิม (v1-v5)**
2. สำรวจคะแนนผู้มอบหมายงานที่สอดคล้องกับงาน (task-aligned assigner score) ของ **YOLO แบบ Anchor-Free ยุคใหม่ (v8-v11)**
3. ทำความเข้าใจว่าการปรับเปลี่ยนเกณฑ์ความมั่นใจ (`conf`) ส่งผลต่อ **ความแม่นยำ (Precision), การระลึก (Recall), ผลบวกจริง (TP), ผลบวกลวง (FP) และผลลบลวง (FN)** อย่างไร
4. สร้างสูตรเหล่านี้จากศูนย์และแสดงภาพพฤติกรรมของพวกมัน

มาเริ่มกันเลย!

---

## 📖 1. โครงสร้างของคะแนนความมั่นใจ

### A. YOLO ดั้งเดิม (v1 - v5)
ในสถาปัตยกรรมแบบ anchor-based แบบคลาสสิก เครือข่ายจะทำนายกล่องขอบเขตและความน่าจะเป็นของการมีความเป็นวัตถุ ("objectness") แยกต่างหาก คะแนนความมั่นใจเฉพาะคลาสสุดท้ายจะคำนวณได้ดังนี้:

$$\text{Class Score} = P(\text{Class}_i | \text{Object}) \times P(\text{Object}) \times \text{IoU}_{\text{pred}}^{\text{truth}}$$

โดยที่:
- $P(\text{Class}_i | \text{Object})$: ความน่าจะเป็นแบบมีเงื่อนไขที่วัตถุนั้นเป็นคลาส $i$ เมื่อพบว่ามีวัตถุอยู่จริง
- $P(\text{Object})$: **คะแนนความเป็นวัตถุ (objectness score)** (ความน่าจะเป็นที่เซลล์กริด/สมอนั้นมีวัตถุใด ๆ อยู่ข้างใน)
- $\text{IoU}_{\text{pred}}^{\text{truth}}$: ค่าจุดตัดส่วนด้วยจุดรวม (Intersection over Union) ระหว่างกล่องที่ทำนายและกล่องจริง (ground truth)

การคูณสามส่วนนี้ช่วยให้มั่นใจว่าการตรวจจับจะมีความมั่นใจสูงก็ต่อเมื่อเครือข่ายแน่ใจว่ามีวัตถุอยู่ แน่ใจในคลาสของวัตถุ และระบุตำแหน่งของมันได้อย่างแม่นยำ

### B. YOLO แบบ Anchor-Free (v8 - v11)
สถาปัตยกรรม YOLO ยุคใหม่ (v8, v9, v10, v11) ตัดส่วนแยกของการทำนายความเป็นวัตถุ (objectness branch) ออกไปทั้งหมด เพื่อลดความหน่วง (latency) และทำให้ loss landscape ง่ายขึ้น แทนที่จะใช้ส่วนหัวแยกต่างหาก พวกมันจะทำนายความน่าจะเป็นของคลาส ($s$) โดยตรงสำหรับแต่ละสมอ ในระหว่างการฝึกฝนและการมอบหมายเป้าหมาย พวกมันจะจัดแนวคุณภาพการจำแนกประเภทและการระบุตำแหน่งให้เป็นคะแนนเดียวกัน $t$ โดยใช้ **Task-Aligned Assigner**:

$$t = s^\alpha \times \text{IoU}^\beta$$

โดยที่:
- $s$ คือความน่าจะเป็นของคลาสที่ทำนายได้
- $\text{IoU}$ คือพื้นที่ทับซ้อนเชิงพื้นที่กับกล่องจริง (ground-truth box)
- $\alpha$ และ $\beta$ คือปัจจัยการถ่วงน้ำหนักที่ควบคุมอิทธิพลของการจำแนกประเภทและการระบุตำแหน่ง ใน Ultralytics YOLOv8/v11 ค่าเริ่มต้นกำหนดให้ $\alpha = 0.5$ และ $\beta = 6.0$

ค่าที่สูงของ $\beta=6.0$ จะลดทอนคะแนนสำหรับกล่องทำนายที่มีพื้นที่ทับซ้อนต่ำอย่างมาก บังคับให้แบบจำลองจัดลำดับความสำคัญของความแม่นยำในขอบเขตกล่อง

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calculate_classic_confidence(p_class_given_obj, p_obj, iou):
    """Calculates classic YOLO class score: P(Class|Obj) * P(Obj) * IoU"""
    return p_class_given_obj * p_obj * iou

def calculate_task_aligned_score(s, iou, alpha=0.5, beta=6.0):
    """Calculates task-aligned score: s^alpha * IoU^beta"""
    return (s ** alpha) * (iou ** beta)

## 🔬 2. การปรับเปลี่ยนเกณฑ์: การชดเชยระหว่าง Precision และ Recall (Precision-Recall Trade-off)

มาทบทวนคำจำกัดความของตัวชี้วัดหลักของเราจากหลักการพื้นฐานกัน:
- **ผลบวกจริง (True Positive - TP)**: การคาดการณ์ที่ระบุวัตถุจริงได้อย่างถูกต้อง (คะแนนความมั่นใจ $\ge$ เกณฑ์ และ $\text{IoU} \ge \text{IoU\_Threshold}$)
- **ผลบวกลวง (False Positive - FP)**: การคาดการณ์ที่ไม่ถูกต้อง (คะแนนความมั่นใจ $\ge$ เกณฑ์ และ $\text{IoU} < \text{IoU\_Threshold}$)
- **ผลลบลวง (False Negative - FN)**: วัตถุจริงที่*ไม่*ถูกตรวจพบ (ไม่มีการคาดการณ์ใดแมปเข้ากับวัตถุจริงด้วยคะแนนความมั่นใจ $\ge$ เกณฑ์ และ $\text{IoU} \ge \text{IoU\_Threshold}$)

### คำจำกัดความทางคณิตศาสตร์:
$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
*จากวัตถุทั้งหมดที่ทำนายออกมา มีความถูกต้องกี่เปอร์เซ็นต์?*

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{\text{TP}}{\text{Total Ground Truths}}$$
*จากวัตถุจริงทั้งหมดในรูปภาพ เราสามารถหาพบกี่เปอร์เซ็นต์?*

### การปรับเปลี่ยนค่าเกณฑ์ (\tau):
- **เกณฑ์สูง (เช่น 0.8)**: เข้มงวดมาก เราจะเก็บเฉพาะการทำนายที่เครือข่ายมีความมั่นใจสูงมากเท่านั้น สิ่งนี้จะช่วยลดผลบวกลวง (ทำให้ Precision สูง) แต่ก็จะทำให้เราพลาดวัตถุไปหลายชิ้นด้วย (ทำให้ Recall ต่ำ, ผลลบลวงสูง)
- **เกณฑ์ต่ำ (เช่น 0.1)**: ผ่อนปรนมาก เราเก็บเกือบทุกอย่าง สิ่งนี้จะช่วยเพิ่มผลบวกจริง (ทำให้ Recall สูง, ผลลบลวงต่ำ) แต่ก็จะเกิดสัญญาณเตือนที่ผิดพลาดจำนวนมาก (ทำให้ Precision ต่ำ, ผลบวกลวงสูง)

In [ ]:
def evaluate_predictions(predictions, num_ground_truths, conf_threshold, iou_threshold=0.5):
    """Evaluates TP, FP, FN, Precision, and Recall at a specific threshold."""
    filtered_preds = [p for p in predictions if p['score'] >= conf_threshold]
    
    tp = sum(1 for p in filtered_preds if p['iou'] >= iou_threshold)
    fp = sum(1 for p in filtered_preds if p['iou'] < iou_threshold)
    fn = num_ground_truths - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0  # Standard choice when no predictions are made
    recall = tp / num_ground_truths if num_ground_truths > 0 else 0.0
    
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall
    }

## 📊 3. การจำลองและการแสดงผลการปรับเปลี่ยนเกณฑ์ด้วยภาพ

มาจำลองชุดข้อมูลที่มีการทำนาย 120 รายการเทียบกับวัตถุจริง 60 รายการกัน เราจะเปลี่ยนค่าเกณฑ์ความมั่นใจตั้งแต่ 0.0 ถึง 1.0 และสังเกตการเปลี่ยนแปลงของตัวชี้วัดต่าง ๆ

In [ ]:
np.random.seed(42)
num_gt = 60

# Simulate 120 predictions
# Scores uniformly distributed
scores = np.random.uniform(0.05, 0.99, 120)
# IoUs around a bell curve centered at 0.55
ious = np.clip(np.random.normal(0.55, 0.22, 120), 0.0, 1.0)

simulated_predictions = [{"score": s, "iou": i} for s, i in zip(scores, ious)]

# Evaluate over a range of thresholds
thresholds = np.linspace(0.0, 1.0, 101)
tps, fps, fns = [], [], []
precisions, recalls = [], []

for t in thresholds:
    metrics = evaluate_predictions(simulated_predictions, num_gt, t)
    tps.append(metrics["TP"])
    fps.append(metrics["FP"])
    fns.append(metrics["FN"])
    precisions.append(metrics["Precision"])
    recalls.append(metrics["Recall"])

tps = np.array(tps)
fps = np.array(fps)
fns = np.array(fns)
precisions = np.array(precisions)
recalls = np.array(recalls)

In [ ]:
# Plotting the results using Matplotlib
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Precision & Recall vs Threshold
axes[0].plot(thresholds, precisions, label='Precision', color='#1f77b4', linewidth=2.5)
axes[0].plot(thresholds, recalls, label='Recall', color='#ff7f0e', linewidth=2.5)
axes[0].set_xlabel('Confidence Threshold (conf)', fontsize=12)
axes[0].set_ylabel('Metric Value', fontsize=12)
axes[0].set_title('Precision & Recall vs. Threshold', fontsize=14, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=11)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# Plot 2: TP, FP, FN Counts vs Threshold
axes[1].plot(thresholds, tps, label='True Positives (TP)', color='#2ca02c', linewidth=2.5)
axes[1].plot(thresholds, fps, label='False Positives (FP)', color='#d62728', linewidth=2.5)
axes[1].plot(thresholds, fns, label='False Negatives (FN)', color='#9467bd', linewidth=2.5)
axes[1].set_xlabel('Confidence Threshold (conf)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('TP, FP, FN Counts vs. Threshold', fontsize=14, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=11)
axes[1].set_xlim(0, 1)

# Plot 3: Precision-Recall Curve
axes[2].plot(recalls, precisions, color='#8c564b', linewidth=3)
axes[2].set_xlabel('Recall', fontsize=12)
axes[2].set_ylabel('Precision', fontsize=12)
axes[2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].set_xlim(0, 1.05)
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()